**Traffic Anomaly Detection — Isolation Forest**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

This project detects traffic anomalies directed toward a specific site using WAF logs. The baseline is segmented by shift (Morning / Night) and Device Action (Allowed, Alerted, Blocked), so each hourly event count is compared only against previous hours that share the same shift and action context — reducing false positives caused by natural traffic variation across time windows.

To improve prediction quality, the Isolation Forest contamination parameter is set dynamically rather than using a fixed value. The contamination rate is derived from the proportion of data points that exceed the dataset's upper bound (calculated via IQR), meaning the model adapts its anomaly budget to the actual distribution of each segment.

After scoring each hourly count using the decision_function, results are filtered to retain only the lowest (most negative) scores, which correspond to the highest-traffic outliers. This intentionally excludes negative anomalies — hours with abnormally low counts — since the focus is on detecting traffic spikes rather than drops.


In [ ]:
df=pd.read_csv("/content/NEWgen22 (2).csv")
df['Time [UTC]']=pd.to_datetime(df['Time'])

results=[]

for site in df['site'].unique():
 for shift in df['shift'].unique():
  for Action in df['Action'].unique():
   df_site=df[(site==df['site']) & (shift==df['shift']) & (Action==df['Action'])].copy()
   df_site=df_site.sort_values('Time [UTC]')


   X=df_site[['count']]
   Scaler=StandardScaler()
   X_scaled=Scaler.fit_transform(X)
   means=df_site['count'].mean()

   df_site['avarege']=means
   q25=df_site['count'].quantile(0.25)
   q75=df_site['count'].quantile(0.75)
   IQR=q75-q25
   upper_bound=q75+1.5*IQR
   containment=(df_site['count']>=upper_bound).mean()
   print(f'upper bound is:{upper_bound}')

   containment=np.clip(containment,0.01,0.5)
   model=IsolationForest(n_estimators=50,contamination='auto',max_samples="auto",max_features=1.0,bootstrap=False,n_jobs=None,random_state=None,warm_start=False)
   model.fit(X_scaled)
   df_site['anomaly']=model.predict(X_scaled)
   df_site['anomaly']= df_site['anomaly'].map({1:0,-1:1})
   df_site['score'] = model.decision_function(X_scaled)
   results.append(df_site)
   df_final=pd.concat(results)
   df_final.to_excel(f"anomalies{site}-{shift}-{Action}-11.xlsx")
   df_anomalies=df_final[(df_final['anomaly']==1)& (df_final['site']==site) & (shift==df['shift']) & (Action==df['Action'])& (df_final['score']<-0.23357)]
   df_anomalies.to_excel(f"anomalies{site}-{shift}-{Action}-34.xlsx")
   print(df_anomalies)

   df_site=df_final[(df_final['site']==site) & (shift==df['shift']) & (Action==df['Action'])]
   plt.figure(figsize=(12,6))
   plt.plot(df_site['Time [UTC]'],df_site['count'],label='traffic')
   plt.scatter(df_anomalies['Time [UTC]'],df_anomalies['count'],color='red',label='anomalies')
   plt.title(f'anomaly detection-{site}-{shift}-{Action}')
   plt.xlabel('Time [UTC]')
   plt.ylabel('traffic')
   plt.legend()
   plt.show()
   results=[]
   df_final=[]